In [1]:
import json
import re

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

with open("../../backend/data/data-v2/train-v2.json", "r", encoding='utf-8') as f:
    train = json.load(f)
with open('../../backend/data/data-v2/validation-v2.json', 'r', encoding = 'utf-8') as f:
    validation = json.load(f)
with open('../../backend/data/data-v2/test-v2.json', 'r', encoding='utf-8') as f:
    test = json.load(f)

print(len(train))
print(len(test))
print(len(validation))

print(train[100])
print(type(train))

3672
500
500
{'text': 'Uống vitamin dẻo nhai rộp rộp khen ngon.', 'label': 'activities'}
<class 'list'>


In [2]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[.!?,;:]', '', text)
    return text.split()

In [3]:
from collections import Counter

counter = Counter()
for item in train:
    tokens = tokenizer(item['text'])
    counter.update(tokens)


In [4]:
max_vocab_size = 20000
vocab = {
    '<pad>':0,
    '<unk>':1
}
for word, _ in counter.most_common(max_vocab_size):
    vocab[word] = len(vocab)
print(len(vocab))

3156


In [5]:
print("Number of samples:", len(train))
print("Vocab size:", len(vocab))
print(train[0])
print(train[1])
print(train[2])

Number of samples: 3672
Vocab size: 3156
{'text': 'Bé nhà mình nhút nhát thấy rõ, cứ thấy đông người là nép chặt vào chân cô không rời.', 'label': 'mood'}
{'text': 'Chiều mát ra bãi cỏ chơi đu quay, không may con bị con ong vàng chích trúng ngay vành tai sưng vù, đỏ ửng. Cô đã lập tức vạch tìm rút ngòi nọc ra, bôi kem chống dị ứng nên bé bớt khóc và chịu đi rửa tay.', 'label': 'health'}
{'text': 'Em bé đang tuổi mọc răng hàm nên lợi sưng viêm tấy đỏ, hành sốt hầm hập suốt cữ sáng. Em lười nhai, chỉ thích ngậm, cô đành phải hâm nóng lại bát cháo, xay nhuyễn ra đút từng muỗng nhỏ xíu thì con mới ăn được lưng bát.', 'label': 'health'}


In [6]:
all_tokens = []

for item in train:
    tokens = item["text"].split()
    all_tokens.extend(tokens)

freq = Counter(all_tokens)

print("Number of tokens:", len(all_tokens))
print("Vocab:", len(freq))
print("Freq = 1:", sum(1 for x in freq.values() if x == 1))
print("Freq <= 2:", sum(1 for x in freq.values() if x <= 2))

Number of tokens: 72031
Vocab: 5700
Freq = 1: 1792
Freq <= 2: 2635


In [7]:
def numericalize(text):
    tokens = tokenizer(text)
    return[
        vocab.get(token, vocab[('<unk>')])
        for token in tokens
    ]
text = "trưa nay bé Kiệt ăn không hết phần cơm, ngủ ít"
numericalized = numericalize(text)
print(numericalized)

[26, 10, 5, 1, 9, 7, 64, 89, 40, 18, 386]


In [8]:
label_to_id = {
    'meals' : 0,
    'sleeping' : 1,
    'health' : 2,
    'activities' : 3,
    'mood' : 4
}

In [9]:
class kindergatern_dataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        item = self.dataset[index]
        text = numericalize(item['text'])
        label = label_to_id[item['label']]
        return torch.tensor(text), torch.tensor(label)

In [10]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.stack(labels)
    return texts, labels
train_dataset = kindergatern_dataset(train)
validation_dataset = kindergatern_dataset(validation)
test_dataset = kindergatern_dataset(test)
print(type(train_dataset))
print(type(train_dataset.dataset))
print(type(train_dataset.dataset[0]))
print(train_dataset.dataset[0])

<class '__main__.kindergatern_dataset'>
<class 'list'>
<class 'dict'>
{'text': 'Bé nhà mình nhút nhát thấy rõ, cứ thấy đông người là nép chặt vào chân cô không rời.', 'label': 'mood'}


In [11]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    collate_fn = collate_fn
)

validation_dataloader = DataLoader(
    validation_dataset,
    batch_size = 32,
    shuffle = False,
    collate_fn = collate_fn
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False,
    collate_fn = collate_fn
)

In [12]:
texts, labels = next(iter(train_dataloader))
print(texts.shape)
print(labels.shape)
embedding_dim = 128

torch.nn.Embedding(
    max_vocab_size, embedding_dim
)

torch.Size([32, 71])
torch.Size([32])


Embedding(20000, 128)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(). __init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx = 0
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        embedd = self.embedding(x)
        out_put, (hidden, _) = self.lstm(embedd)
        hidden = hidden[-1]
        out_put = self.fc(hidden)
        return out_put

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim = embedding_dim,
    hidden_dim = 64,
    num_classes = 5
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

cuda


In [22]:
num_epoch = 20
for epoch in range(num_epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for texts, labels in train_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out_put = model(texts)
        loss = criterion(out_put, labels)
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()
        predictions = out_put.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / len(train_dataloader)
    accuracy = correct / total * 100
    print(
        f"Epoch {epoch + 1}/{num_epoch} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch 1/20 | Loss: 1.6050 | Accuracy: 21.98%
Epoch 2/20 | Loss: 1.5957 | Accuracy: 23.23%
Epoch 3/20 | Loss: 1.5814 | Accuracy: 24.35%
Epoch 4/20 | Loss: 1.5625 | Accuracy: 25.25%
Epoch 5/20 | Loss: 1.5179 | Accuracy: 30.28%
Epoch 6/20 | Loss: 1.4790 | Accuracy: 33.69%
Epoch 7/20 | Loss: 1.4863 | Accuracy: 33.74%
Epoch 8/20 | Loss: 1.5567 | Accuracy: 27.78%
Epoch 9/20 | Loss: 1.4752 | Accuracy: 33.55%
Epoch 10/20 | Loss: 1.4130 | Accuracy: 38.18%
Epoch 11/20 | Loss: 1.4010 | Accuracy: 37.17%
Epoch 12/20 | Loss: 1.3604 | Accuracy: 40.58%
Epoch 13/20 | Loss: 1.3972 | Accuracy: 38.10%
Epoch 14/20 | Loss: 1.3689 | Accuracy: 39.49%
Epoch 15/20 | Loss: 1.3699 | Accuracy: 40.25%
Epoch 16/20 | Loss: 1.3427 | Accuracy: 42.05%
Epoch 17/20 | Loss: 1.3074 | Accuracy: 45.32%
Epoch 18/20 | Loss: 1.2908 | Accuracy: 46.21%
Epoch 19/20 | Loss: 1.2952 | Accuracy: 45.40%
Epoch 20/20 | Loss: 1.3045 | Accuracy: 44.72%


In [25]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in validation_dataloader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.3900


In [27]:
from collections import defaultdict

confusion_matrix = torch.zeros(5, 5, dtype=torch.int64)

model.eval()

with torch.no_grad():
    for texts, labels in test_dataloader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)
        predictions = outputs.argmax(dim=1)

        for true_label, predicted_label in zip(labels, predictions):
            confusion_matrix[
                true_label.item(),
                predicted_label.item()
            ] += 1

print("\n===== CONFUSION MATRIX =====")

labels = [
    "meals",
    "sleeping",
    "health",
    "activities",
    "mood"
]

print(f"{'Actual ↓ / Predicted →':<20}", end="")

for label in labels:
    print(f"{label:<12}", end="")

print()

for i, label in enumerate(labels):
    print(f"{label:<20}", end="")

    for j in range(5):
        print(f"{confusion_matrix[i][j].item():<12}", end="")

    print()


===== CONFUSION MATRIX =====
Actual ↓ / Predicted →meals       sleeping    health      activities  mood        
meals               67          3           1           29          0           
sleeping            42          31          0           27          0           
health              66          3           14          17          0           
activities          18          0           0           81          1           
mood                22          2           1           75          0           
